In [6]:
from dotenv import load_dotenv

_ = load_dotenv()

## Setup Tools

In [2]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "travel_server": {
            "transport": "streamable_http",
            "url": "https://mcp.kiwi.com"
        }
    }
)

tools = await client.get_tools()

In [3]:
from typing import Dict, Any
from tavily import TavilyClient
from langchain.tools import tool

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for relevant information."""
    return tavily_client.search(query)

In [4]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

@tool
def query_playlist_db(query: str) -> str:
    """Query the database for playlist information."""
    try:
        return db.run(query)
    except Exception as e:
        return f"Error querying database: {e}"

## Create State

In [5]:
from langchain.agents import AgentState

class WeddingState(AgentState):
    origin: str
    destination: str
    guest_count: str
    genre: str

## Create Subagents

In [7]:
from langchain.agents import create_agent

travel_agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
    system_prompt="""
    You are a travel agent. Search for flights to the desired destination wedding location.
    You are not allowed to ask any more follow up questions, you must find the bes flight
    options based on the following criteria:
    - Price (lowest, economy class)
    - Duration (shortest)
    - Date (time of year which you believe is best for a wedding at this location) 
    To make thins easy, only look for one ticket, one way.
    You may need to make multiple searches to interatively find the best options.
    You wil be given no extra information, only the origin and destination. It is your
    job to think critically about the best options.
    Once you have found the best options, let the user know your shortlist of options.
    """
)

In [8]:
venue_agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search],
    system_prompt="""
    You are a venue specialist. Search for venues in the desired location, and with the
    desired capacity.
    You are not allowed to ask any more follow up questions, you must find the beste venue
    options based on the following criteria:
    - Price (lowest)
    - Capacity (exact match)
    - Reviews (highest)
    You may need to make multiple searches to iteratively find the best options.
    """
)

In [9]:
dj_agent = create_agent(
    model="gpt-5-nano",
    tools=[query_playlist_db],
    system_prompt="""
    You are a playlist specialist. Query the sql database and curate the perfect playlist
    for a wedding given a genre.
    Once you have your playlist, calculate the total duration and cost of the playlist,
    each song has an associated price.
    If you run into errors when querying the database, try to fix them by making changes
    to the query.
    Do not come back empty handed, keep trying to query the db until you find a list of
    songs.
    You may need to make multiple queries to iteratively find the best options.
    """ 
)

## Cordinator

In [11]:
from langchain.tools import ToolRuntime
from langchain.messages import HumanMessage, ToolMessage
from langgraph.types import Command

@tool
async def search_flights(runtime: ToolRuntime) -> str:
    """Travel agent searches for flights to the desired destination wedding location."""
    origin = runtime.state["origin"]
    destination = runtime.state["destination"]
    response = await travel_agent.ainvoke(
        {"messages": [HumanMessage(content=f"Find flights from {origin} to {destination}.")]}
    )
    return response["messages"][-1].content

@tool
def search_venues(runtime: ToolRuntime) -> str:
    """Venue agent chooses the best venue for the giving location and capacity."""
    destination = runtime.state["destination"]
    capacity = runtime.state["guest_count"]
    query = f"Find wedding venues in {destination} for {capacity} guests."
    response = venue_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response["messages"][-1].content

@tool
def suggest_playlist(runtime: ToolRuntime) -> str:
    """Playlist agent curates the perfect playlist for the given genre."""
    genre = runtime.state["genre"]
    query = f"Find {genre} tracks for wedding playlist."
    response = dj_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response["messages"][-1].content

@tool
def update_state(origin: str, destination: str, guest_count: str, genre: str, runtime: ToolRuntime) -> str:
    """Update the state when you know all of the values: origin, destination, guest_count, genre."""
    return Command(
        update={
            "origin": origin,
            "destination": destination,
            "guest_count": guest_count,
            "genre": genre,
            "messages": [ToolMessage(content="Successfully updated state.", tool_call_id=runtime.tool_call_id)]
        }
    )


In [12]:
coordinator_agent = create_agent(
    model="gpt-5-nano",
    tools=[search_flights, search_venues, suggest_playlist, update_state],
    state_schema=WeddingState,
    system_prompt="""
    You are a wedding coordinator. Delegate tasks to your specialists for flights, venues,
    and playlists.
    First, find all the information you need to update the state. Once that is done you
    can delegate the tasks.
    Onde you have received their answers, coordinate the perfect wedding for me.
    """
)

## Test

In [13]:
response = await coordinator_agent.ainvoke(
    {"messages": [HumanMessage(content="I'm from London and I'd like a wedding in Paris for 10 guests, jazz-genre")]}
)

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

I'm from London and I'd like a wedding in Paris for 10 guests, jazz-genre
================================== Ai Message ==================================
Tool Calls:
  update_state (call_ffQOGJwDeREDsANpBEaoFxBT)
 Call ID: call_ffQOGJwDeREDsANpBEaoFxBT
  Args:
    origin: London
    destination: Paris
    guest_count: 10
    genre: jazz
================================= Tool Message =================================
Name: update_state

Successfully updated state.
================================== Ai Message ==================================
Tool Calls:
  search_flights (call_kBdmrmb08Bnw6Q6Lo7KY6o85)
 Call ID: call_kBdmrmb08Bnw6Q6Lo7KY6o85
  Args:
  search_venues (call_PkCxQ5yeL72bZJJCZH8cWHyz)
 Call ID: call_PkCxQ5yeL72bZJJCZH8cWHyz
  Args:
  suggest_playlist (call_JSdn1gR4K5vraIAjNF42abfF)
 Call ID: call_JSdn1gR4K5vraIAjNF42abfF
  Args:
================================= Tool Message =================

In [14]:
print(response["messages"][-1].content)

Fantastic choice—Paris with a jazz vibe for 10 guests sounds beautiful. Here’s how I’ll coordinate it and what I’ve lined up so far. I’ve also delegated tasks to our specialists to move this forward quickly.

What we’ve found so far (summary from specialists)
- Flights (London to Paris, for a small group)
  - Several direct London to Paris options (CDG), with a mix of cheapest, shortest duration, and good-value times.
  - Example options:
    - Budget: Luton → CDG, 13 May 2026, 06:00–08:25, 51 EUR per person. Nonstop.
    - Budget: Luton → CDG, 13 May 2026, 11:15–13:40, 51 EUR per person. Nonstop.
    - Shortest duration: Gatwick or Stansted to CDG, around 70 minutes total (examples around 07:00–10:20 or 16:55–18:15), prices around 59–65 EUR.
  - Notes: If you share your preferred date window, I’ll lock in 2–3 best group options and request group pricing and transfers from the airport to your venue/hotel.
- Venues for exactly 10 guests (Paris)
  - Eiffel Tower Port Debilly (near the Ei